# Optical Soliton History & Modulation: 1973–2001

Derive the physics. Understand why solitons almost won — and why coherent DWDM beat them instead.

In [ ]:
from sympy import *
from IPython.display import display, Markdown
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

init_printing(use_latex='mathjax')

def step(label, expr=None):
    display(Markdown(f'**{label}**'))
    if expr is not None:
        display(expr)

def section(title):
    display(Markdown(f'---\n## {title}'))

## §1 — Timeline: Soliton from Theory to Abandonment

In [ ]:
section('Soliton chronology 1834–2001')

display(Markdown(r'''
| Year | Event | Who |
|------|-------|-----|
| 1834 | First water soliton observed, Scottish canal | Scott Russell |
| 1965 | "Soliton" coined, KdV equation numerics | Kruskal & Zabusky |
| **1973** | **Theoretical prediction: optical fiber soliton** | **Hasegawa & Tappert** |
| 1980 | First experimental optical soliton, 700m fiber | Mollenauer (Bell Labs) |
| 1988 | Soliton propagation with Raman amplification, 4000 km | Mollenauer |
| 1991 | Dispersion-managed (DM) soliton concept | Knox, Bergman, Stolen |
| 1995 | 10 Gb/s soliton across Atlantic (simulated route) | Bell Labs |
| 1999 | 160 Gb/s DM soliton lab demo | NTT |
| **2001** | **Telecom bubble collapses; coherent DWDM wins commercially** | Industry |
| 2010s | Soliton microcombs (Kerr) revive soliton physics on-chip | Kippenberg lab |

**Why 2001 matters**: September 11 + telecom crash froze capital.
DWDM with EDFAs was already deployed. Soliton networks required
dispersion-managed fiber — a forklift upgrade nobody funded.
'''))

## §2 — NLSE: The Governing Equation

In [ ]:
section('Nonlinear Schrödinger Equation (NLSE)')

display(Markdown(r'''
The NLSE governs pulse propagation in a single-mode fiber:

$$i\frac{\partial A}{\partial z} - \frac{\beta_2}{2}\frac{\partial^2 A}{\partial t^2} + \gamma|A|^2 A = 0$$

- $A(z,t)$: slowly-varying envelope amplitude [W^{1/2}]
- $\beta_2$: group-velocity dispersion [ps²/km] — anomalous when $\beta_2 < 0$
- $\gamma = n_2 \omega_0 / (c A_{eff})$: nonlinear coefficient [W⁻¹km⁻¹]

**Three forces in tension**:
1. Dispersion alone → pulse broadens as $t(z) = T_0\sqrt{1+(z/L_D)^2}$
2. SPM alone → chirp accumulates, spectrum broadens
3. **Balanced**: soliton — shape-preserving propagation
'''))

# Characteristic lengths
T0, beta2, gamma_nl, P0 = symbols('T_0 beta_2 gamma P_0', positive=True)

L_D = T0**2 / beta2
L_NL = 1 / (gamma_nl * P0)

step('Dispersion length L_D = T₀²/|β₂| =', L_D)
step('Nonlinear length L_NL = 1/(γP₀) =', L_NL)

display(Markdown(r'''
**Soliton condition**: $L_D = L_{NL}$
$$\frac{T_0^2}{|\beta_2|} = \frac{1}{\gamma P_0} \implies P_0 = \frac{|\beta_2|}{\gamma T_0^2}$$

This is the *fundamental soliton* (N=1). Higher-order solitons have N²×more power.
'''))

# SMF-28 numbers
beta2_smf = 17e-3   # ps²/m at 1550nm (normal dispersion side; anomalous = -17)
# anomalous for solitons:
beta2_anom = 17e-3  # |β₂| in ps²/m
gamma_smf = 1.3e-3  # W⁻¹m⁻¹
T0_ps = 1.0         # 1 ps pulse
T0_s  = T0_ps * 1e-12

P0_soliton = beta2_anom / (gamma_smf * T0_ps**2)  # in W (T0 in ps, β₂ in ps²/m → P in W)
L_D_val = T0_ps**2 / beta2_anom  # in m

print(f'SMF-28 at 1550 nm, T₀=1 ps:')
print(f'  |β₂| = {beta2_anom*1e3:.1f} ps²/km  (anomalous dispersion)')
print(f'  γ    = {gamma_smf*1e3:.1f} W⁻¹km⁻¹')
print(f'  P₀ (fundamental soliton) = {P0_soliton:.1f} W')
print(f'  L_D = T₀²/|β₂| = {L_D_val:.0f} m')

## §3 — Exact Soliton Solution: sech

In [ ]:
section('Fundamental soliton: A(z,t) = A₀ sech(t/T₀) exp(iz/2L_D)')

display(Markdown(r'''
The exact N=1 soliton solution to the NLSE is:

$$A(z,t) = A_0 \,\text{sech}\!\left(\frac{t}{T_0}\right) \exp\!\left(\frac{iz}{2L_D}\right)$$

- **Amplitude**: $A_0 = \sqrt{P_0}$
- **Phase**: accumulates linearly with z — just a global phase, no chirp
- **Shape**: sech envelope is **chirp-free** — dispersion and SPM cancel exactly
- **Power spectrum**: $|\tilde{A}(\omega)|^2 \propto \text{sech}^2(\pi\omega T_0/2)$ — also sech!
'''))

t_sym, z_sym, T0_sym, L_D_sym, A0_sym = symbols('t z T_0 L_D A_0', real=True)

A_soliton = A0_sym * sech(t_sym/T0_sym) * exp(I*z_sym/(2*L_D_sym))
step('Soliton A(z,t) =', A_soliton)

# Intensity profile
I_profile = simplify(A_soliton * conjugate(A_soliton))
step('|A(z,t)|² = A₀² sech²(t/T₀) =', I_profile)

# FWHM of sech²
display(Markdown(r'''
**FWHM of sech²**: solve sech²(t/T₀) = 1/2
$$t_{FWHM} = 2T_0 \cosh^{-1}(\sqrt{2}) \approx 1.7627\,T_0$$
'''  ))

fwhm_factor = 2*acosh(sqrt(2))
step('FWHM factor =', fwhm_factor, )
print(f'Numeric: FWHM = {float(fwhm_factor):.4f} × T₀')

# Plot
t_vals = np.linspace(-5, 5, 500)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(t_vals, 1/np.cosh(t_vals)**2, 'b-', linewidth=2, label='sech²(t/T₀)')
axes[0].axhline(0.5, color='r', linestyle='--', label='FWHM level')
axes[0].axvline(float(fwhm_factor)/2, color='g', linestyle=':', label=f'T_FWHM/2={float(fwhm_factor)/2:.3f}T₀')
axes[0].axvline(-float(fwhm_factor)/2, color='g', linestyle=':')
axes[0].set_xlabel('t/T₀')
axes[0].set_ylabel('Intensity (normalized)')
axes[0].set_title('Soliton intensity: sech²')
axes[0].legend(fontsize=8)

# Spectrum: FT of sech is also sech
omega_vals = np.linspace(-10, 10, 500)
# FT{sech(t/T0)} = π T0 sech(π ω T0/2)
axes[1].plot(omega_vals, (np.pi/np.cosh(np.pi*omega_vals/2))**2, 'r-', linewidth=2)
axes[1].set_xlabel('ωT₀')
axes[1].set_ylabel('|Ã(ω)|² (normalized)')
axes[1].set_title('Soliton spectrum: also sech²')

plt.tight_layout()
plt.savefig('soliton_profile.png', dpi=100)
plt.show()
print('Soliton is self-dual: same shape in time and frequency domains')

## §4 — Modulation Formats: RZ vs NRZ vs DPSK

In [ ]:
section('Modulation format evolution: 1970s → 2001')

display(Markdown(r'''
### Format timeline

| Era | Format | Spectral eff. | Notes |
|-----|--------|---------------|-------|
| 1975–1985 | NRZ-OOK | 0.4 b/s/Hz | Simple, fiber loss limits to ~100 km |
| 1985–1995 | RZ-OOK | 0.4 b/s/Hz | Shorter pulses → compatible with soliton regime |
| 1991–2001 | DM-Soliton | 0.4–0.8 | Dispersion managed; 10 Gb/s transoceanic |
| 1995–2001 | DWDM + EDFA | 0.8+ | 80×10G = 800G on one fiber, won commercially |
| 2000+ | DPSK, DQPSK | 1–2 | Phase keying, better sensitivity |
| 2010+ | DP-16QAM | 8 b/s/Hz | Coherent detection, DSP compensation |

### Why solitons lost to DWDM
1. **Gordon-Haus jitter**: ASE noise from EDFAs randomly shifts soliton timing → BER floor
2. **Soliton–soliton interaction**: adjacent pulses attract/repel → pattern-dependent errors
3. **Dispersion management infrastructure cost**: every fiber span must be engineered
4. **EDFA + DWDM already deployed** by 1995–2001 at massive scale
5. **Coherent DSP** (post-2005) can correct dispersion in silicon → no special fiber needed
'''))

# Gordon-Haus timing jitter
sigma_t, D_s, n_sp, hbar_sym, omega_s, P0_s, T0_s_sym, L_amp, L_total = symbols(
    'sigma_t D_s n_sp hbar omega_s P_0 T_0 L_amp L_total', positive=True)

display(Markdown(r'''
### Gordon-Haus timing jitter (RMS)

$$\sigma_t^2 = \frac{n_{sp} \hbar \omega_s}{3 P_0 T_0} \cdot |\beta_2|^2 \cdot \frac{L_{total}^3}{L_{amp}^2}$$

Scales as $L^3$ — catastrophic for transoceanic links.
Solution was *sliding-frequency guiding filters* (Mollenauer, 1992) or
synchronous modulation — both complex to deploy.
'''))

## §5 — Dispersion-Managed Soliton

In [ ]:
section('Dispersion-managed (DM) soliton: average vs local dispersion')

display(Markdown(r'''
**Key idea**: alternate fiber segments with opposite dispersion.
Average dispersion $\langle\beta_2\rangle$ near zero, but local $|\beta_2|$ large.

$$\langle\beta_2\rangle = \frac{\beta_{2,+} L_+ + \beta_{2,-} L_-}{L_+ + L_-} \approx 0$$

The DM soliton:
- Is **not** an exact sech — it breathes within each map period
- Has **enhanced peak power** vs fundamental soliton
- Reduces Gordon-Haus jitter (smaller $\langle\beta_2\rangle$)
- Reduces four-wave mixing between WDM channels (large local $|\beta_2|$)
'''))

# Map strength
beta2p, beta2m, Lp, Lm = symbols('beta_{2+} beta_{2-} L_+ L_-', real=True)
T0_DM = symbols('T_0', positive=True)

# Map strength S = (β₂+ L+ - β₂- L-)/(T₀²) — characterizes breathing
display(Markdown(r'''
**Map strength** (dimensionless):
$$S = \frac{|\beta_{2+}|L_+ + |\beta_{2-}|L_-}{T_0^2}$$

- $S < 1$: quasi-linear DM soliton (weakly nonlinear)
- $S \approx 1$: optimal for capacity
- $S > 5$: strongly managed, large breathing

Commercial transoceanic links (e.g., TAT-12, FLAG, SEA-ME-WE) targeted $S \approx 0.3$–$1$.
'''))

# Numeric example
b2_anom = -17.0  # ps²/km, anomalous
b2_norm =  3.5   # ps²/km, normal (DCF-like)
L_anom  = 90.0   # km
L_norm  = 10.0   # km  (DCF is shorter, high β₂)
T0_val  = 10.0   # ps

avg_b2 = (b2_anom*L_anom + b2_norm*L_norm*(-1)) / (L_anom + L_norm)  # DCF has opposite sign
# actually DCF has β₂ > 0 (normal, same as positive in our convention below)
b2_dcf = 100.0  # ps²/km, normal DCF is strongly normal
L_dcf  = 2.89   # km to compensate

avg_b2_correct = (b2_anom*L_anom + b2_dcf*L_dcf) / (L_anom + L_dcf)
S_val = (abs(b2_anom)*L_anom + abs(b2_dcf)*L_dcf) / T0_val**2

print(f'DM map: {L_anom}km SMF (β₂={b2_anom} ps²/km) + {L_dcf}km DCF (β₂=+{b2_dcf} ps²/km)')
print(f'Average β₂ = {avg_b2_correct:.3f} ps²/km')
print(f'Map strength S (T₀={T0_val}ps) = {S_val:.2f}')
print(f'Period = {L_anom+L_dcf:.1f} km')

## §6 — IEEE & NIST Standards for Optical Communications

In [ ]:
section('IEEE 802.3 / ITU-T G-series / NIST SP 250 — measurement standards')

display(Markdown(r'''
### IEEE 802.3 (Ethernet over fiber)

| Standard | Speed | Fiber | Reach | Notes |
|----------|-------|-------|-------|-------|
| 802.3z (1998) | 1 GbE | SMF/MMF | 550m–5km | First fiber GbE |
| 802.3ae (2002) | 10 GbE | SMF | 10–40 km | 10GBASE-LR/ER |
| 802.3ba (2010) | 40/100 GbE | SMF | 10–40 km | First 100G standard |
| 802.3bs (2017) | 200/400 GbE | SMF | 500m–10km | DP-16QAM |
| 802.3ck (2022) | 800 GbE | SMF | 2km | Current |

### ITU-T G-series (long-haul, carrier)

| Standard | Topic |
|----------|-------|
| G.652 | Standard SMF (SMF-28: α=0.2dB/km, β₂=−17ps²/km at 1550nm) |
| G.654 | Ultra-low-loss SMF (α=0.15dB/km, submarine) |
| G.655 | NZ-DSF: non-zero dispersion-shifted, D=2–6 ps/nm/km |
| G.694.1 | DWDM channel grid: 100GHz, 50GHz, 25GHz spacing |
| G.709 | OTN optical transport network framing |
| G.975.1 | FEC for submarine, BER 10⁻¹³ after coding |

### NIST SP 250 — measurement traceability

NIST Special Publication 250 series defines calibration services for:
- Optical power meters (SP 250-54): NIST-traceable W, dBm
- Fiber geometry (SP 250-59): core diameter, NA, MFD
- Chromatic dispersion (SP 250-76): ps/nm/km, measurement uncertainty
- Polarization mode dispersion (PMD): ps/√km

**For Jalali lab**: NIST-traceable calibration needed for TDGSA intensity measurements.
Uncalibrated power → systematic phase error that cannot be corrected by more GS iterations.
'''))

# Dispersion units conversion
display(Markdown(r'''
### Unit conversion: β₂ ↔ D

$$D = -\frac{2\pi c}{\lambda^2}\beta_2 \quad [\text{ps}/(\text{nm}\cdot\text{km})]$$

$$\beta_2 = -\frac{D\lambda^2}{2\pi c} \quad [\text{ps}^2/\text{km}]$$
'''))

lam, c_light = symbols('lambda c', positive=True)
D_sym = symbols('D', real=True)

beta2_from_D = -D_sym * lam**2 / (2*pi*c_light)
step('β₂ from D =', beta2_from_D)

# SMF-28 at 1550nm
D_smf28 = 17.0      # ps/(nm·km)
lam_nm  = 1550.0    # nm
c_nmps  = 3e8 * 1e9 / 1e12  # nm/ps = 3e5 nm/ps

beta2_calc = -D_smf28 * lam_nm**2 / (2*np.pi * c_nmps)  # ps²/km
print(f'\nSMF-28: D=+17 ps/(nm·km) at 1550nm')
print(f'β₂ = {beta2_calc:.2f} ps²/km  (anomalous: negative β₂)')
print(f'Matches datasheet: −21.7 ps²/km (small deviation from using D=17 vs actual D=17.5)')

## §7 — Kerr Soliton Microcombs: Soliton Revival on Chip

In [ ]:
section('Dissipative Kerr soliton (DKS) in microresonators, 2010s–present')

display(Markdown(r'''
**Key shift**: soliton physics abandoned in fiber (2001) but revived on chip.

A CW pump laser drives a microring resonator (Si₃N₄ or MgF₂).
Above threshold: optical parametric oscillation → frequency comb.
With correct detuning: DKS forms — a *circulating soliton* inside the resonator.

$$\text{Output: frequency comb} \quad f_n = f_{rep} \cdot n + f_{CEO}$$

where $f_{rep} = c/(n_{eff} \cdot 2\pi R)$ (free spectral range).

### Lugiato-Lefever Equation (LLE) — driven NLSE:

$$\frac{\partial E}{\partial t} = -\left(1 + i\Delta\right)E - i\frac{\beta_2}{2}\frac{\partial^2 E}{\partial\phi^2} + i|E|^2 E + F$$

- $\Delta$: laser-cavity detuning
- $F$: pump field
- $\phi$: round-trip coordinate (periodic)

### Applications of Kerr combs:
| Application | Why soliton combs |
|-------------|-------------------|
| Optical frequency synthesis | Phase-coherent, octave-spanning |
| LiDAR | Mode-locked → precise ranging |
| Optical clocks | CEO frequency stabilization |
| **TS-DFT / TDGSA** | Broadband source at chip scale |
| Coherent DWDM | Each comb line = one WDM channel |
'''))

# FSR of a ring
R_ring = symbols('R', positive=True)
n_eff_sym = symbols('n_eff', positive=True)
c_s = symbols('c', positive=True)

FSR = c_s / (n_eff_sym * 2*pi*R_ring)
step('FSR = c/(n_eff · 2πR) =', FSR)

# Numeric: Si₃N₄ ring, R=100μm
n_eff_SiN = 1.9
R_um = 100e-6  # m
c_ms = 3e8    # m/s
FSR_Hz = c_ms / (n_eff_SiN * 2*np.pi*R_um)
print(f'\nSi₃N₄ ring, R=100μm, n_eff=1.9:')
print(f'  FSR = {FSR_Hz/1e9:.1f} GHz')
print(f'  This sets the comb tooth spacing')
print(f'  Soliton rep rate = FSR = {FSR_Hz/1e9:.1f} GHz (one soliton per round trip)')

display(Markdown(r'''
**Connection to Jalali lab TDGSA**:
A soliton microcomb provides the broadband, coherent, pulsed source
for TS-DFT. The dispersion stretch maps each comb tooth to a
distinct time slot → single-shot spectroscopy at GHz frame rates.
TDGSA recovers the phase of each comb line from two intensity measurements.
'''))

## §8 — TDGSA Connection: Why β₂z ≥ 5000 ps² Is a Soliton-Length Condition

In [ ]:
section('TDGSA diversity requirement from soliton physics')

display(Markdown(r'''
**Recall**: TDGSA requires $|\beta_2 z| \geq 5000\,\text{ps}^2$ for convergence.

This is not arbitrary — it is a *dispersion length* condition:

$$|\beta_2 z| \geq 5000\,\text{ps}^2 \iff z \geq L_D = \frac{T_0^2}{|\beta_2|}$$

For a soliton with $T_0 = 1\,\text{ps}$, $|\beta_2| = 17\,\text{ps}^2/\text{km}$:
$$L_D = \frac{1^2}{17\times 10^{-3}} \approx 59\,\text{m}$$

For TDGSA at $|\beta_2 z| = 5000\,\text{ps}^2$ with SMF-28:
$$z = \frac{5000\,\text{ps}^2}{17\,\text{ps}^2/\text{km}} \approx 294\,\text{km}$$

**Physical interpretation**: the two measurements must be taken far enough apart
in the dispersive domain that they carry *independent* phase information.
If $|\beta_2 z| \ll 5000$, the intensity at $z_1$ and $z_2$ are nearly identical
(correlation $\rho > 0.95$) → GS oscillates without converging.

### In TS-DFT (on-chip):
- SOI spiral: $\beta_2 \approx -1000\,\text{ps}^2/\text{m}$, need $L \geq 5\,\text{m}$
- Achievable with on-chip spiral in $< 1\,\text{mm}^2$ area
- This is the chip design target for Jalali lab integrated TDGSA
'''))

# Sweep: β₂z needed as function of T₀
T0_vals_ps = np.logspace(-1, 2, 200)  # 0.1 ps to 100 ps
# For soliton: β₂z_soliton = T₀² (when L_NL = L_D)
b2z_soliton = T0_vals_ps**2  # ps²

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(T0_vals_ps, b2z_soliton, 'b-', linewidth=2, label='Soliton condition: |β₂z|=T₀²')
ax.axhline(5000, color='red', linestyle='--', linewidth=2, label='TDGSA min: |β₂z|=5000 ps²')
ax.axvline(np.sqrt(5000), color='green', linestyle=':', linewidth=2,
           label=f'T₀={np.sqrt(5000):.0f} ps crossover')
ax.fill_between(T0_vals_ps, 5000, b2z_soliton,
                where=(b2z_soliton > 5000), alpha=0.2, color='green',
                label='TDGSA OK region')
ax.set_xlabel('Pulse duration T₀ (ps)')
ax.set_ylabel('|β₂z| (ps²)')
ax.set_title('TDGSA diversity vs soliton condition')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('tdgsa_soliton_condition.png', dpi=100)
plt.show()
print(f'Crossover: T₀ > {np.sqrt(5000):.0f} ps → TDGSA diversity exceeds soliton condition')
print('Short pulses (T₀ < 70 ps) need EXTRA dispersion beyond soliton length for GS to converge')